# ComfyUI on Colab - SDXL (未实测草稿)

免费 T4 16GB 跑 ComfyUI + SDXL base 1.0。首次约 8-12 分钟：装 ComfyUI、下 6.9GB 模型、起 cloudflared 隧道，拿到 trycloudflare.com 链接后在浏览器里用 ComfyUI。
用完记得断开并删除运行时，不要空转。隧道链接无鉴权，不要公开分享。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -q -r requirements.txt

In [ ]:
# SDXL base 1.0, 约 6.9GB, 只下 base, 不下 refiner
!wget -q --show-progress https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P /content/ComfyUI/models/checkpoints/
!ls -lh /content/ComfyUI/models/checkpoints/

In [ ]:
# cloudflared 隧道, 用来把本地 8188 端口暴露成公网链接
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

In [ ]:
# 后台起 ComfyUI, 再起隧道, 打印 trycloudflare.com 链接
import subprocess, time, re
subprocess.Popen(['python', 'main.py', '--listen', '--port', '8188'], cwd='/content/ComfyUI', stdout=open('/tmp/comfy.log','w'), stderr=subprocess.STDOUT)
time.sleep(20)
print(open('/tmp/comfy.log').read()[-2000:])
!cloudflared tunnel --url http://localhost:8188 2>&1 | grep -o 'https://[^ ]*trycloudflare.com' | head -1